# Forschungsfrage 4 - Semantische Heterogenität: Traditioneller Ansatz (Dictionary/Fuzzy-Mapping)

`thread_category` (55 feingranulare, teils uneinheitliche Werte) wird über eine
**Mehrheitsvotum-Nachschlagetabelle** (thread_category -> häufigste zugehörige
parent_category) auf eine der 12 kanonischen Kategorien abgebildet - ganz ohne
Freitextanalyse des `title`.

**Wichtiger methodischer Punkt:** Die im Rahmen der Benchmark-Erstellung
mitgelieferte Tabelle `tf4_thread_to_parent_mapping.csv` wurde auf **allen**
Zeilen mit bekannter `parent_category` berechnet - also unter Einschluss der
späteren `test`-Zeilen. Für eine faire Bewertung wird die Mehrheitstabelle hier
**ausschließlich auf dem `train`-Split neu berechnet** (Data Leakage vermieden,
analog zum Vorgehen in Forschungsfrage 2, Median/Modus-Notebook).

Für `thread_category`-Werte, die im `train`-Split nicht vorkommen (das betrifft
hier ausschließlich die reale Anwendungsmenge, nicht den Test-Split - siehe
Abschnitt 3), wird **RapidFuzz** zum unscharfen Abgleich mit den bekannten
`thread_category`-Zeichenketten eingesetzt; führt auch das zu keinem
brauchbaren Treffer, wird auf die insgesamt häufigste Kategorie zurückgefallen.
Ein Sonderfall wird zusätzlich behandelt: Ist der unbekannte `thread_category`-Wert
selbst bereits einer der 12 zulässigen `parent_category`-Bezeichnungen (kommt in
der Anwendungsmenge vor), wird er direkt übernommen.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os

from rapidfuzz import fuzz, process
from sklearn.metrics import accuracy_score, f1_score

os.makedirs("results", exist_ok=True)

eval_df = pd.read_csv("benchmark/tf4_semantic_eval.csv")
app_df = pd.read_csv("benchmark/tf4_application_set.csv")

train = eval_df[eval_df["split"] == "train"]
test = eval_df[eval_df["split"] == "test"]

VALID_CATEGORIES = sorted(eval_df["true_parent_category"].unique().tolist())


## 2. Mehrheitsvotum-Tabelle (ausschließlich aus `train`) + globaler Rückfall

In [2]:
majority_map = train.groupby("thread_category")["true_parent_category"].agg(lambda s: s.value_counts().idxmax())
global_majority = train["true_parent_category"].value_counts().idxmax()

print(f"Bekannte thread_category-Werte (train): {len(majority_map)}")
print(f"Globale Rückfall-Kategorie: {global_majority}")

known_thread_categories = list(majority_map.index)


Bekannte thread_category-Werte (train): 40
Globale Rückfall-Kategorie: Computers & Electronics


## 3. Zuordnungsfunktion: exakte Suche -> Sonderfall -> Fuzzy-Suche -> globaler Rückfall

In [3]:
FUZZY_THRESHOLD = 80  # RapidFuzz-Score (0-100)

def map_thread_to_parent(thread_category):
    if pd.isna(thread_category):
        return global_majority, "kein_wert"
    if thread_category in majority_map.index:
        return majority_map[thread_category], "exakt"
    if thread_category in VALID_CATEGORIES:
        return thread_category, "sonderfall_bereits_parent_category"
    match = process.extractOne(thread_category, known_thread_categories, scorer=fuzz.token_sort_ratio)
    if match is not None and match[1] >= FUZZY_THRESHOLD:
        matched_thread_cat = match[0]
        return majority_map[matched_thread_cat], f"fuzzy(score={match[1]:.0f} -> '{matched_thread_cat}')"
    return global_majority, "globaler_rueckfall"


## 4. Bewertung auf dem `test`-Split

In [4]:
t0 = time.time()
test = test.copy()
mapped = test["thread_category"].apply(map_thread_to_parent)
test["parent_category_pred_dict"] = mapped.apply(lambda x: x[0])
test["zuordnungsart"] = mapped.apply(lambda x: x[1])
lookup_time = time.time() - t0

accuracy = accuracy_score(test["true_parent_category"], test["parent_category_pred_dict"])
macro_f1 = f1_score(test["true_parent_category"], test["parent_category_pred_dict"], average="macro")
print(f"Accuracy: {accuracy:.3f}  Macro-F1: {macro_f1:.3f}  (n_test={len(test)}, {lookup_time*1000:.2f}ms)")
print(test["zuordnungsart"].value_counts())

test[["row_id", "thread_category", "true_parent_category", "parent_category_pred_dict", "zuordnungsart"]].to_csv(
    "results/tf4_dictionary_predictions.csv", index=False)


Accuracy: 0.895  Macro-F1: 0.924  (n_test=124, 1.64ms)
zuordnungsart
exakt    124
Name: count, dtype: int64


## 5. Anwendung auf die reale Anwendungsmenge (500 Zeilen mit fehlender `parent_category`)

Hier greift der Fuzzy-Anteil der Methode tatsächlich: Ein Teil der `thread_category`-
Werte in dieser Menge entspricht bereits direkt einer `parent_category`-Bezeichnung
(Sonderfall aus Abschnitt 3) - ein konkretes Beispiel für die im Rohdatensatz
vorkommende semantische Heterogenität (uneinheitliche Granularität derselben
Spalte).

In [5]:
t0 = time.time()
mapped_app = app_df["thread_category"].apply(map_thread_to_parent)
app_out = app_df[["row_id", "thread_category", "title"]].copy()
app_out["parent_category_pred_dict"] = mapped_app.apply(lambda x: x[0])
app_out["zuordnungsart"] = mapped_app.apply(lambda x: x[1])
app_lookup_time = time.time() - t0

print(f"Anwendungsmenge verarbeitet: {len(app_out)} Zeilen in {app_lookup_time*1000:.2f}ms")
print(app_out["zuordnungsart"].value_counts())

app_out.to_csv("results/tf4_dictionary_application_predictions.csv", index=False)


Anwendungsmenge verarbeitet: 500 Zeilen in 3.47ms
zuordnungsart
sonderfall_bereits_parent_category    423
globaler_rueckfall                     77
Name: count, dtype: int64


## 6. Metriken und Laufzeit-Log speichern

In [6]:
metrics = {
    "experiment": "TF4_Semantik", "method": "Dictionary_Fuzzy",
    "accuracy": accuracy, "macro_f1": macro_f1, "n_test": len(test),
    "lookup_time_sec": lookup_time, "n_application_rows": len(app_df),
    "hinweis": "Mehrheitstabelle ausschliesslich aus train-Split berechnet (Data-Leakage-Fix ggue. tf4_thread_to_parent_mapping.csv).",
}
with open("results/tf4_dictionary_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([{
    "experiment": "TF4_Semantik", "method": "Dictionary_Fuzzy", "n_items": len(test),
    "wall_time_sec": lookup_time, "input_tokens": 0, "output_tokens": 0,
    "estimated_cost_usd": 0.0, "model_name": "mehrheitsvotum+rapidfuzz (kein LLM/API)",
}])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print("Gespeichert: results/tf4_dictionary_predictions.csv, results/tf4_dictionary_application_predictions.csv, results/tf4_dictionary_metrics.json")


Gespeichert: results/tf4_dictionary_predictions.csv, results/tf4_dictionary_application_predictions.csv, results/tf4_dictionary_metrics.json
